Entity

In [1]:
from dataclasses import dataclass
from pathlib import Path
import os
 
@dataclass(frozen=True)
class DataPreprocessingConfig:
    root_dir: Path
    unzip_dir: Path
    train_dir: Path
    test_dir: Path
    input_size: int
    resize_size: int
    randaugment_num_ops: int
    randaugment_magnitude: int
    random_erasing_p: float
    seed: int
    batch_size: int
    num_workers: int

In [2]:
os.getcwd()

'd:\\Personal_projects\\Pyhton_proj\\AI-Food-Recognition-Nutrition-Assistant\\research'

In [3]:
os.chdir("..")

In [4]:
%pwd

'd:\\Personal_projects\\Pyhton_proj\\AI-Food-Recognition-Nutrition-Assistant'

Config Manager

In [5]:
from AI_Food_Recognition_Nutrition_Assistant.constants import *
from AI_Food_Recognition_Nutrition_Assistant.utils.common import read_yaml,create_directories

[2026-04-19 14:04:08,694: INFO: dsl_registry: Successfully registered DSL: cutedsl]
[2026-04-19 14:04:08,698: INFO: dsl_registry: Successfully registered DSL: triton]


In [6]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH
                 ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_preprocessing_config(self) -> DataPreprocessingConfig:
        config = self.config.data_preprocessing
        p = self.params
        create_directories([config.root_dir])
        data_preprocessing_config = DataPreprocessingConfig(
            root_dir=Path(config.root_dir),
            unzip_dir=Path(config.unzip_dir),
            train_dir=Path(config.train_dir),
            test_dir=Path(config.test_dir),
            input_size=p.model.input_size,
            resize_size=p.model.resize_size,
            seed = p.training.seed,
            randaugment_num_ops=p.augmentation.randaugment_num_ops,
            randaugment_magnitude=p.augmentation.randaugment_magnitude,
            random_erasing_p=p.augmentation.random_erasing_p,
            batch_size= p.training.batch_size,
            num_workers=p.training.num_workers,
        )

        return data_preprocessing_config

Data Preprocessing Component

In [7]:
from AI_Food_Recognition_Nutrition_Assistant import logger
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch

In [ ]:
class DataPreprocessing:
    """
        Define Transformers and augment them -> train test split
    """
    def __init__(self, config: DataPreprocessingConfig):
        self.config = config
    
    # Define transformers
    def get_train_transform(self) -> transforms.Compose:
        return transforms.Compose([
            transforms.RandomResizedCrop(self.config.input_size),
            transforms.RandomHorizontalFlip(),
            transforms.RandAugment(
                num_ops=self.config.randaugment_num_ops,
                magnitude=self.config.randaugment_magnitude
            ),
            transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225]),
            transforms.RandomErasing(
                p=self.config.random_erasing_p,
                scale=(0.02, 0.2),
                ratio=(0.3, 3.3)
            ),
        ])
 
    def get_val_test_transform(self) -> transforms.Compose:
        return transforms.Compose([
            transforms.Resize(self.config.resize_size),
            transforms.CenterCrop(self.config.input_size),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225]),
        ])
 
    def load_and_split(self):
        train_transform = self.get_train_transform()
        val_test_transform = self.get_val_test_transform()

        dataset = datasets.ImageFolder(root=self.config.unzip_dir)

        logger.info(f"Classes found: {dataset.classes}")
        logger.info(f"Total images: {len(dataset)}")

        # define size
        train_size = int(0.8 * len(dataset))
        val_size = int(0.1 * len(dataset))
        test_size = len(dataset) - train_size - val_size
        
        # get seed and train-test split
        seed_num = torch.Generator().manual_seed(self.config.seed)
        train_data,val_data,test_data = random_split(dataset,lengths=[train_size,val_size,test_size],generator=seed_num)

        # transform and augment
        train_data.dataset.transform = train_transform
        val_data.dataset.transform = val_test_transform
        test_data.dataset.transform = val_test_transform
        print(train_data.dataset is dataset)
        print(train_data.dataset is val_data.dataset)   
        print(val_data.dataset is test_data.dataset)    

        # Define train and test loaders
        train_loader = DataLoader(
            train_data, batch_size=self.config.batch_size,
            shuffle=True, num_workers=self.config.num_workers,
            pin_memory=True, prefetch_factor=4,persistent_workers=True if self.config.num_workers > 0 else False
        )
        val_loader = DataLoader(
            val_data, batch_size=self.config.batch_size,
            shuffle=False, num_workers=self.config.num_workers,
            pin_memory=True, prefetch_factor=4,persistent_workers=True if self.config.num_workers > 0 else False
        )
        test_loader = DataLoader(
            test_data, batch_size=self.config.batch_size,
            shuffle=False, num_workers=self.config.num_workers,
            pin_memory=True, prefetch_factor=4,persistent_workers=True if self.config.num_workers > 0 else False
        )
        logger.info(f"DataLoaders ready. Train: {train_size}, Validation: {val_size}, Test: {test_size}")
        return train_loader,val_loader,test_loader

Pipeline

In [11]:
try:
    config = ConfigurationManager()
    data_preprocessing_config = config.get_data_preprocessing_config()
    preprocessing = DataPreprocessing(config=data_preprocessing_config)
    train_loader,val_loader,test_loader = preprocessing.load_and_split()
except Exception as e:
    raise e

[2026-04-19 14:04:35,691: INFO: common: yaml file: <_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-19 14:04:35,694: INFO: common: yaml file: <_io.TextIOWrapper name='params.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-04-19 14:04:35,696: INFO: common: created directory at: artifacts]
[2026-04-19 14:04:35,697: INFO: common: created directory at: artifacts/data_preprocessing]
[2026-04-19 14:04:35,922: INFO: 1091501018: Classes found: ['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare', 'beet_salad', 'beignets', 'bibimbap', 'bread_pudding', 'breakfast_burrito', 'bruschetta', 'caesar_salad', 'cannoli', 'caprese_salad', 'carrot_cake', 'ceviche', 'cheese_plate', 'cheesecake', 'chicken_curry', 'chicken_quesadilla', 'chicken_wings', 'chocolate_cake', 'chocolate_mousse', 'churros', 'clam_chowder', 'club_sandwich', 'crab_cakes', 'creme_brulee', 'croque_madame', 'cup_cakes', 'deviled_eggs', 'donuts', '